# Week 16, Multi-Agent Triage Team (A/B vs. the single agent)

```text
# Requirements: pip install langgraph openai
```

> ⚠️ REQUIRES: `OPENAI_API_KEY` or `OPENROUTER_API_KEY` (for the real-model classifier hook). With no key, a deterministic keyword classifier drives the same graph.
> ⚠️ REQUIRES: `langgraph` installed. If missing, a manual runner walks the same supervisor + specialist nodes.

Build a **supervisor + three specialists** (tracking, refunds, docs) in LangGraph, route 10 real tickets from `zoro.data.support_tickets()`, then **A/B test the team against the Week 15 single agent** on accuracy, latency, tokens, and cost, and write the architecture justification from the numbers.

## When multi-agent is worth it (and when it isn't)

Multi-agent buys **context isolation**, **role specialization**, **parallelism**, and **failure isolation**, but it *costs* coordination overhead, latency, tokens, and a new failure mode: bad handoffs. Anthropic's rule: start with one well-prompted agent plus good tools, add workflows, and reach for multiple agents only when the benefits clearly outweigh the cost. This notebook measures that tradeoff on real tickets, so the justification is a table, not a slogan.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
_root = pathlib.Path.cwd()
while not (_root / "zoro").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import json, os, re, time
from typing import TypedDict
import numpy as np
from zoro import data

SEED = 42
rng = np.random.default_rng(SEED)

HAS_LG = False
try:
    from langgraph.graph import StateGraph, START, END
    HAS_LG = True
    print("langgraph imported OK")
except Exception:
    print("langgraph absent; manual runner will walk the same nodes.")

_ship = data.shipments(n=50_000, seed=SEED)
_lanes = data.lanes(seed=11)
_car = data.carriers(seed=7)
_ship_by_id = {row.shipment_id: row for row in _ship.itertuples()}
_lane_by_id = {row.lane_id: row for row in _lanes.itertuples()}
_car_by_id = {row.carrier_id: row for row in _car.itertuples()}
_policies = {d["doc_id"]: d for d in data.policy_docs()}

def _est(text):
    return max(1, len(str(text)) // 4)

# Ground truth: ticket category -> specialist route.
CAT_TO_ROUTE = {"tracking": "tracking", "refund": "refunds", "damage": "refunds", "billing": "refunds", "documents": "docs", "customs": "docs"}
print("tools + mapping ready")

## Tools (one set per specialist, plus a shared compute)

Each specialist gets *its own* tool so its context stays small, that is the isolation argument. The single agent, by contrast, must carry every tool.

In [ ]:
def track_shipment(shipment_id):
    sid = str(shipment_id).strip().upper()
    row = _ship_by_id.get(sid)
    if row is None:
        return {"error": "shipment " + sid + " not found"}
    lane = _lane_by_id.get(row.lane_id)
    carrier = _car_by_id.get(row.carrier_id)
    return {"shipment_id": sid, "status": row.status, "carrier": carrier.carrier_name if carrier else row.carrier_id,
            "origin": lane.origin if lane else "?", "destination": lane.destination if lane else "?",
            "delay_hours": round(float(row.delay_hours), 1), "on_time": bool(row.is_on_time)}

def get_policy(doc_id):
    doc = _policies.get(str(doc_id).strip().upper())
    return {"doc_id": doc["doc_id"], "title": doc["title"], "text": doc["text"]} if doc else {"error": "no policy"}

def compute_refund(shipment_id):
    row = _ship_by_id.get(str(shipment_id).strip().upper())
    if row is None:
        return 0.0
    delay, value = float(row.delay_hours), float(row.value_usd)
    if delay > 7 * 24:
        return round(value * 0.50, 2)
    if delay > 48:
        return round(value * 0.10, 2)
    return 0.0

def extract_sid(text):
    m = re.search(r"S\d{7}", str(text), re.I)
    return m.group(0).upper() if m else ""

def supervisor_classify(text):
    # Deterministic classifier (swap for an LLM call to use the real model).
    low = str(text).lower()
    if "refund" in low or "damaged" in low or "damage" in low or "invoice" in low or "wrong weight" in low or "billing" in low:
        return "refunds"
    if "bill of lading" in low or "document" in low or "customs" in low:
        return "docs"
    if "where is" in low or "track" in low or "arrive" in low or "status" in low or "late" in low:
        return "tracking"
    return "escalate"

print("supervisor_classify example:", supervisor_classify("I want a refund for shipment S0000012. It arrived 30 hours late."))

## Specialist + supervisor nodes

Nodes are plain `(state) -> partial state` functions, exactly like Week 15. `supervisor` routes; each `*_specialist` calls its own tool; `synthesize` writes the final answer.

In [ ]:
class TeamState(TypedDict):
    ticket_text: str
    shipment_id: str
    route: str
    specialist_output: str
    final_answer: str
    history: list

def _log(state, entry):
    return list(state.get("history", [])) + [entry]

def supervisor_node(state):
    route = supervisor_classify(state.get("ticket_text", ""))
    return {"route": route, "shipment_id": extract_sid(state.get("ticket_text", "")), "history": _log(state, "supervisor->" + route)}

def tracking_specialist(state):
    out = json.dumps(track_shipment(state.get("shipment_id")))
    return {"specialist_output": out, "history": _log(state, "tracking_specialist->" + out[:50])}

def refunds_specialist(state):
    amount = compute_refund(state.get("shipment_id"))
    out = json.dumps({"refund_amount_usd": amount, "policy": get_policy("POL-002")["text"][:120]})
    return {"specialist_output": out, "history": _log(state, "refunds_specialist->" + out[:50])}

def docs_specialist(state):
    out = json.dumps(get_policy("POL-002"))
    return {"specialist_output": out, "history": _log(state, "docs_specialist->" + out[:50])}

def synthesize_node(state):
    answer = "Resolved by " + state.get("route", "?") + " specialist: " + state.get("specialist_output", "")[:120]
    return {"final_answer": answer, "history": _log(state, "synthesize")}

def escalate_node(state):
    return {"final_answer": "Escalated to a human agent.", "history": _log(state, "escalate")}

def route_map(state):
    return state.get("route", "escalate")

def team_initial(ticket_text, shipment_id=""):
    return {"ticket_text": ticket_text, "shipment_id": shipment_id, "route": "", "specialist_output": "", "final_answer": "", "history": []}

print("team nodes defined")

## Assemble the team graph

Supervisor → (conditional edge) → specialist → synthesize → end, with an escalate branch. Same shape as Week 15, one more specialist per branch.

In [ ]:
graph = None
if HAS_LG:
    g = StateGraph(TeamState)
    g.add_node("supervisor", supervisor_node)
    g.add_node("tracking", tracking_specialist)
    g.add_node("refunds", refunds_specialist)
    g.add_node("docs", docs_specialist)
    g.add_node("synthesize", synthesize_node)
    g.add_node("escalate", escalate_node)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", route_map, {"tracking": "tracking", "refunds": "refunds", "docs": "docs", "escalate": "escalate"})
    g.add_edge("tracking", "synthesize")
    g.add_edge("refunds", "synthesize")
    g.add_edge("docs", "synthesize")
    g.add_edge("synthesize", END)
    g.add_edge("escalate", END)
    graph = g.compile()
    try:
        print(graph.get_graph().draw_mermaid())
    except Exception:
        print("(draw_mermaid unavailable)")

def team_run(ticket_text, shipment_id=""):
    # Walk the same nodes in the same order (works with or without langgraph).
    state = team_initial(ticket_text, shipment_id)
    t0 = time.perf_counter()
    state = {**state, **supervisor_node(state)}
    r = state["route"]
    if r == "tracking":
        state = {**state, **tracking_specialist(state)}
    elif r == "refunds":
        state = {**state, **refunds_specialist(state)}
    elif r == "docs":
        state = {**state, **docs_specialist(state)}
    else:
        state = {**state, **escalate_node(state)}
    state = {**state, **synthesize_node(state)}
    latency = time.perf_counter() - t0
    tokens = sum(_est(h) for h in state["history"]) + 200  # +200 = supervisor handoff + synthesis overhead
    return {"route": state["route"], "final_answer": state["final_answer"], "latency": latency, "tokens": tokens}

print("team_run ready (graph demo + manual runner)")

## The single-agent baseline (Week 15)

One agent carries **all** tools and does classify → tool → answer with no handoff and no synthesis, the conservative default the team has to beat.

In [ ]:
def single_run(ticket_text, shipment_id=""):
    t0 = time.perf_counter()
    route = supervisor_classify(ticket_text)
    sid = extract_sid(ticket_text)
    if route == "tracking":
        out = json.dumps(track_shipment(sid))
    elif route == "refunds":
        out = json.dumps({"refund_amount_usd": compute_refund(sid), "policy": get_policy("POL-002")["text"][:120]})
    elif route == "docs":
        out = json.dumps(get_policy("POL-002"))
    else:
        out = "escalated"
    answer = "Resolved: " + out[:120]
    latency = time.perf_counter() - t0
    tokens = _est(ticket_text) + _est(out) + _est(answer)
    return {"route": route, "final_answer": answer, "latency": latency, "tokens": tokens}

print("single_run ready")

## A/B on 10 real tickets

Route the same 10 tickets through both systems and collect accuracy, latency, tokens, and cost (tokens × a price per 1M). The cost model is illustrative but the *shape* of the comparison is what matters.

In [ ]:
tickets = data.support_tickets(n=50, seed=99)
ab_tickets = tickets.head(10)

INPUT_PM, OUTPUT_PM = 2.50, 10.00  # USD per 1M tokens (illustrative)

def cost_of(tokens):
    return tokens / 1e6 * OUTPUT_PM

rows = []
for t in ab_tickets.itertuples():
    gt = CAT_TO_ROUTE.get(t.category, "escalate")
    tm = team_run(t.text, t.shipment_id)
    sg = single_run(t.text, t.shipment_id)
    rows.append({
        "ticket": t.ticket_id,
        "category": t.category,
        "gt": gt,
        "team_route": tm["route"],
        "single_route": sg["route"],
        "team_latency_ms": round(tm["latency"] * 1000, 2),
        "single_latency_ms": round(sg["latency"] * 1000, 2),
        "team_tokens": tm["tokens"],
        "single_tokens": sg["tokens"],
        "team_cost": round(cost_of(tm["tokens"]), 6),
        "single_cost": round(cost_of(sg["tokens"]), 6),
    })

team_correct = sum(1 for r in rows if r["team_route"] == r["gt"])
single_correct = sum(1 for r in rows if r["single_route"] == r["gt"])
team_acc = round(team_correct / len(rows), 4)
single_acc = round(single_correct / len(rows), 4)

print("ticket   category   gt        team       single")
for r in rows:
    print("%-8s %-10s %-9s %-10s %s" % (r["ticket"], r["category"], r["gt"], r["team_route"], r["single_route"]))

print()
print("--- A/B report (n=%d) ---" % len(rows))
print("accuracy        team %.4f  single %.4f" % (team_acc, single_acc))
print("avg latency(ms) team %.2f  single %.2f" % (sum(r["team_latency_ms"] for r in rows) / len(rows), sum(r["single_latency_ms"] for r in rows) / len(rows)))
print("total tokens    team %d  single %d" % (sum(r["team_tokens"] for r in rows), sum(r["single_tokens"] for r in rows)))
print("total cost ($)  team %.6f  single %.6f" % (sum(r["team_cost"] for r in rows), sum(r["single_cost"] for r in rows)))

In [ ]:
# Architecture justification, written from the numbers above.
team_cost = sum(r["team_cost"] for r in rows)
single_cost = sum(r["single_cost"] for r in rows)
team_tokens = sum(r["team_tokens"] for r in rows)
single_tokens = sum(r["single_tokens"] for r in rows)

justification = (
    "On these 10 tickets the team and the single agent route with equal accuracy "
    f"({team_acc} vs {single_acc}), but the team spends {team_tokens - single_tokens} more tokens "
    f"(+${round(team_cost - single_cost, 6)}) on supervisor handoffs and synthesis with no accuracy gain. "
    "The team is therefore NOT justified for this slice: the intents are easy to classify, the tools are cheap, "
    "and nothing needs isolated context or parallel work. The team earns its cost only when specialists need "
    "separate context windows, per-specialist tools, or parallel execution on tickets that one prompt cannot hold."
)
print(justification)

In [ ]:
print("TEAM_ACCURACY", team_acc)
print("SINGLE_ACCURACY", single_acc)
print("ACCURACY_DELTA", round(team_acc - single_acc, 4))